In [1]:
!pip uninstall -y dgl torchdata

!pip install -q torch==2.2.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

# Install torchdata compatible with torch 2.2
!pip install -q torchdata==0.7.1

# Install DGL for cu121
!pip install dgl -q -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html

# Install DGL LifeSci
!pip install -q dgllife rdkit pandas

Found existing installation: torchdata 0.11.0
Uninstalling torchdata-0.11.0:
  Successfully uninstalled torchdata-0.11.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 2.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 169.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 232.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 228.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 62.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 77.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 147.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 179.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 157.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 132.6 MB/s et

In [2]:
%%writefile /kaggle/working/data_loader.py
import torch
import dgl
from torch.utils.data import Dataset, DataLoader, random_split
from dgllife.utils import ScaffoldSplitter


class MoleculeDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


class MoleculeDatasetWithSmiles(Dataset):
    """Wraps (graph, label, smiles) tuples. Exposes .smiles for dgllife's ScaffoldSplitter."""
    def __init__(self, data):
        self.data   = data
        self.smiles = [item[2] for item in data]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        g, y, smiles = self.data[idx]
        return g, y


def collate_fn(batch):
    graphs, labels = zip(*batch)
    batched_graph  = dgl.batch(graphs)
    batched_labels = torch.stack(labels, dim=0).squeeze(1)
    return batched_graph, batched_labels


# ===========================================================================
#  TOX21 — SCAFFOLD SPLIT
# ===========================================================================
def get_tox21_scaffold_loaders(path="/kaggle/input/datasets/prajwalnayakat/3d-tox21-with-smiles-and-one-hot/tox21_3d_egnn_dataset_scaffold_onehot.pt", batch_size=32):
    raw     = torch.load(path, weights_only=False)   # list of (g, y, smiles)
    dataset = MoleculeDatasetWithSmiles(raw)

    train_set, val_set, test_set = ScaffoldSplitter.train_val_test_split(
        dataset,
        frac_train=0.8,
        frac_val=0.1,
        frac_test=0.1
    )

    print(f"Tox21 Scaffold Split → Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    return train_loader, val_loader, test_loader


# ===========================================================================
#  QM9 — RANDOM SPLIT (unchanged, consistent with literature)
# ===========================================================================
# def get_qm9_loaders(path=r"/kaggle/input/datasets/prajwalnayakat/molecolyte-datasets/qm9_3d_egnn_dataset.pt", batch_size=32):
#     raw     = torch.load(path, weights_only=False)
#     dataset = MoleculeDataset(raw)

#     total = len(dataset)
#     train_size = int(0.8 * total)
#     val_size   = int(0.1 * total)
#     test_size  = total - train_size - val_size

#     print(f"QM9 Split → Train: {train_size} | Val: {val_size} | Test: {test_size}")

#     train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

#     train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  collate_fn=collate_fn)
#     val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
#     test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

#     return train_loader, val_loader, test_loader


tox21_train_loader, tox21_val_loader, tox21_test_loader = get_tox21_scaffold_loaders()
# qm9_train_loader,   qm9_val_loader,   qm9_test_loader   = get_qm9_loaders()

Writing /kaggle/working/data_loader.py


In [3]:
%%writefile /kaggle/working/egnn_layer.py
import torch
import torch.nn as nn
import dgl
import dgl.function as fn


class EGNNLayer(nn.Module):
    """
    One layer of an Equivariant Graph Neural Network (EGNN).
    Based on: "E(n) Equivariant Graph Neural Networks" (Satorras et al., 2021)

    What makes EGNN different from GINEConv:
    - GINEConv: uses pre-computed distances as extra edge features. Coordinates
      are static — they never change during message passing.
    - EGNN: computes distances live from pos during every forward pass, AND
      updates the 3D coordinates of every node as part of the layer itself.
      This means the geometry evolves as information flows through the network,
      making it sensitive to the actual 3D shape of the molecule.

    Per-layer operations:
    1. For every edge: compute distance from current pos, run edge MLP
    2. For every node: aggregate neighbour messages, run node MLP → new hidden state
    3. For every node: compute a weighted sum of relative position vectors → update pos
    """

    def __init__(self, hidden_dim, edge_attr_dim=5):
        super().__init__()

        # Edge MLP: takes [h_i, h_j, distance, edge_attr] → message
        # hidden_dim * 2 for the two node states + 1 for distance + edge_attr_dim for bond type
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1 + edge_attr_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

        # Node MLP: takes [h_i, aggregated messages] → new h_i
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Coordinate MLP: takes edge message → scalar weight for pos update
        # Output is a single scalar that scales the relative position vector (pos_i - pos_j)
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def edge_message(self, edges):
        """
        Runs on every edge simultaneously.
        Computes the distance between the two endpoint atoms from their
        current positions, then feeds everything into the edge MLP.
        """
        # Relative position vector and its scalar distance
        rel_pos  = edges.src['pos'] - edges.dst['pos']          # (E, 3)
        distance = torch.sqrt((rel_pos ** 2).sum(dim=-1, keepdim=True) + 1e-8)    # (E, 1)

        # Concatenate: source hidden state, dest hidden state, distance, bond type
        edge_input = torch.cat([
            edges.src['h'],           # (E, hidden_dim)
            edges.dst['h'],           # (E, hidden_dim)
            distance,                  # (E, 1)
            edges.data['edge_attr'],   # (E, edge_attr_dim)
        ], dim=-1)

        message    = self.edge_mlp(edge_input)     # (E, hidden_dim)
        coord_weight = self.coord_mlp(message)     # (E, 1) — scalar for pos update

        return {
            'message':      message,
            'coord_weight': coord_weight * rel_pos,  # (E, 3) — weighted relative vector
        }

    def node_update(self, nodes):
        """
        Runs on every node simultaneously.
        Aggregates incoming messages and updates the node's hidden state.
        """
        # 'agg_msg' is the sum of all incoming messages (set by dgl after edge_message)
        node_input = torch.cat([nodes.data['h'], nodes.data['agg_msg']], dim=-1)
        new_h      = self.node_mlp(node_input)
        return {'h': new_h}

    def forward(self, g, h, pos, edge_attr):
        with g.local_scope():
            g.ndata['h']         = h
            g.ndata['pos']       = pos
            g.edata['edge_attr'] = edge_attr

            # Step 1: compute messages and coordinate weights along every edge
            g.apply_edges(self.edge_message)

            # Step 2: aggregate messages into each node
            g.update_all(fn.copy_e('message', 'm'), fn.sum('m', 'agg_msg'))

            # Step 3: update node hidden states
            g.apply_nodes(self.node_update)

            # Step 4: update coordinates
            # Sum the weighted relative vectors arriving at each node
            g.update_all(fn.copy_e('coord_weight', 'cw'), fn.sum('cw', 'agg_cw'))
            new_pos = pos + g.ndata['agg_cw']   # (N, 3)

            return g.ndata['h'], new_pos


Writing /kaggle/working/egnn_layer.py


In [4]:
%%writefile /kaggle/working/train_Tox21.py
import time
import torch
import torch.nn as nn
import torch.optim as optim
import dgl
from sklearn.metrics import roc_auc_score
from egnn_layer import EGNNLayer
from data_loader import tox21_train_loader, tox21_val_loader


torch.manual_seed(2026)


# ==========================================
# 1. MODEL ARCHITECTURE
# ==========================================
class MoleColyteEGNN(nn.Module):
    def __init__(self, in_node_features=20, hidden_dim=128, edge_attr_dim=5, num_layers=3, out_features=1):
        super().__init__()

        self.input_proj = nn.Linear(in_node_features, hidden_dim)

        self.egnn_layers = nn.ModuleList([
            EGNNLayer(hidden_dim=hidden_dim, edge_attr_dim=edge_attr_dim)
            for _ in range(num_layers)
        ])

        self.prediction_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, out_features)
        )

    def forward(self, g, x, pos, edge_attr):
        h = self.input_proj(x)

        for layer in self.egnn_layers:
            h, pos = layer(g, h, pos, edge_attr)

        g.ndata['h'] = h
        mol_embedding = dgl.mean_nodes(g, 'h')

        return self.prediction_head(mol_embedding)


# ==========================================
# 2. DYNAMIC WEIGHT CALCULATION
# ==========================================
def calculate_dynamic_weights(loader):
    print("Scanning training data to calculate exact imbalance penalties...")
    num_pos = torch.zeros(12)
    num_neg = torch.zeros(12)

    for batch_graph, batch_labels in loader:
        targets = batch_labels.to(torch.float)

        for i in range(12):
            col       = targets[:, i]
            valid_col = col[col == col]  

            num_pos[i] += (valid_col == 1).sum()
            num_neg[i] += (valid_col == 0).sum()

    dynamic_weights = num_neg / (num_pos + 1e-5)
    print(f"Calculated Pathway Penalties: {dynamic_weights.cpu().tolist()}\n")
    return dynamic_weights.clamp(max=15.0)


# ==========================================
# 3. AUC CALCULATION HELPER
# ==========================================
def compute_mean_auc(all_preds, all_targets):
    """
    all_preds, all_targets: torch tensors of shape (N, 12), already sigmoid-applied for preds.
    Returns mean AUC-ROC across all 12 assays, skipping assays with only one class present.
    Stays numpy-free (Kaggle's numpy has been unreliable) — uses .tolist() for sklearn.
    """
    aucs = []
    for i in range(12):
        valid_mask   = all_targets[:, i] == all_targets[:, i]  # NaN mask
        task_targets = all_targets[valid_mask, i].tolist()
        task_preds   = all_preds[valid_mask,   i].tolist()

        if len(set(task_targets)) > 1:
            aucs.append(roc_auc_score(task_targets, task_preds))

    return sum(aucs) / len(aucs) if aucs else 0.0


# ==========================================
# 4. FINE-TUNING LOOP
# ==========================================
def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Kiln: {device}")

    # Load the pre-trained ToxCast model
    model = MoleColyteEGNN(out_features=599)

    toxcast_state_dict = torch.load(
        "/kaggle/input/models/prajwalnayakat/egnn-on-toxcast-with-one-hot/pytorch/default/4/egnn_toxcast_scaffold_supervised_best.pt",
        weights_only=True,
        map_location=device
    )
    model.load_state_dict(toxcast_state_dict)
    print("ToxCast pretrained weights loaded successfully.")

    # Swap the prediction head for Tox21's 12 binary targets
    model.prediction_head = nn.Sequential(
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 12)
    )
    model = model.to(device)

    # ------------------------------------------
    # LEARNING RATE DIFFERENTIALS
    # ------------------------------------------
    pretrained_lr = 1e-5  # Low LR to protect ToxCast 3D weights
    head_lr       = 1e-3  # High LR to train the new Tox21 classification head

    optimizer = optim.Adam([
        {'params': model.input_proj.parameters(),  'lr': pretrained_lr},
        {'params': model.egnn_layers.parameters(), 'lr': pretrained_lr},
        {'params': model.prediction_head.parameters(), 'lr': head_lr}
    ])
    dynamic_penalty = calculate_dynamic_weights(tox21_train_loader).to(device)
    criterion       = nn.BCEWithLogitsLoss(pos_weight=dynamic_penalty, reduction='none')

    EPOCHS   = 200
    best_auc = -1.0

    for epoch in range(EPOCHS):
        # ------------------------------------------
        # TRAINING PHASE
        # ------------------------------------------
        model.train()
        total_train_loss = 0
        print(f"Starting Epoch {epoch + 1}")

        for step, (batch_graph, batch_labels) in enumerate(tox21_train_loader):
            batch_graph  = batch_graph.to(device)
            batch_labels = batch_labels.to(device)

            x         = batch_graph.ndata['x'].to(torch.float)
            pos       = batch_graph.ndata['pos'].to(torch.float)
            edge_attr = batch_graph.edata['edge_attr'].to(torch.float)

            optimizer.zero_grad()

            predictions  = model(batch_graph, x, pos, edge_attr)   # (batch, 12)
            target_flags = batch_labels.to(torch.float)             # (batch, 12)

            is_valid = target_flags == target_flags                  # NaN mask
            safe_targets = torch.where(is_valid, target_flags, torch.zeros_like(target_flags))
            raw_loss = criterion(predictions, safe_targets)
            loss     = raw_loss[is_valid].mean()

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(tox21_train_loader)
        print(f"==> Epoch {epoch + 1} Train Complete | Average BCE Loss: {avg_train_loss:.4f}")

        # ------------------------------------------
        # VALIDATION PHASE
        # ------------------------------------------
        model.eval()
        total_val_loss = 0
        all_val_preds   = []
        all_val_targets = []

        with torch.no_grad():
            for batch_graph, batch_labels in tox21_val_loader:
                batch_graph  = batch_graph.to(device)
                batch_labels = batch_labels.to(device)

                x         = batch_graph.ndata['x'].to(torch.float)
                pos       = batch_graph.ndata['pos'].to(torch.float)
                edge_attr = batch_graph.edata['edge_attr'].to(torch.float)

                predictions  = model(batch_graph, x, pos, edge_attr)
                target_flags = batch_labels.to(torch.float)

                is_valid = target_flags == target_flags
                safe_targets = torch.where(is_valid, target_flags, torch.zeros_like(target_flags))
                raw_loss = criterion(predictions, safe_targets)
                loss     = raw_loss[is_valid].mean()

                total_val_loss += loss.item()

                # Accumulate for AUC computation — stay in torch, .cpu() only
                all_val_preds.append(torch.sigmoid(predictions).cpu())
                all_val_targets.append(target_flags.cpu())

        avg_val_loss = total_val_loss / len(tox21_val_loader)

        all_val_preds   = torch.cat(all_val_preds,   dim=0)
        all_val_targets = torch.cat(all_val_targets, dim=0)
        avg_val_auc     = compute_mean_auc(all_val_preds, all_val_targets)

        print(f"\tValidation Phase | Avg Val Loss: {avg_val_loss:.4f} | Avg Val AUC: {avg_val_auc:.4f}")

        # ------------------------------------------
        # CHECKPOINTING — now based on AUC, not loss
        # ------------------------------------------
        if avg_val_auc > best_auc:
            best_auc = avg_val_auc
            torch.save(model.state_dict(), "/kaggle/working/egnn_tox21_scaffold_OH_best.pt")
            print(f"🏆 New best Tox21 model saved! (Highest Val AUC: {best_auc:.4f})\n")
        else:
            print(f"Model did not improve. Best Val AUC remains: {best_auc:.4f}\n")


if __name__ == "__main__":
    since = time.time()
    train()
    print(f"Time: {(time.time() - since) / 60:.2f} minutes.")


Writing /kaggle/working/train_Tox21.py


In [5]:
!python train_Tox21.py


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/kaggle/working/train_Tox21.py", line 2, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 1471, in <module>
    from .functional import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/functional.py", line 9, in <module>
    import torch.nn.functional as F
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/__init__.py", line 1, in <module>
    from .modules import *  # noqa: F403
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/